# YorubaLlama -- Fine-Tune on Validated Yoruba Health QA Data

**Read this before running.** This fine-tunes `Jacaranda/YorubaLlama` (Llama-3-8B, continually pretrained on 8.1GB of real Yoruba text + instruction-tuned on 66,280 Yoruba pairs) on our own validated health Q&A data.

**Why this model:** a real zero-shot check (see `kaggle_yorubaLlama_zeroshot_check.ipynb`) showed it already produces genuinely coherent, real Yoruba -- the first model in this whole search to do so. It also showed REAL but INCONSISTENT health knowledge: a good, accurate answer about diabetes symptoms, but also a wrong answer that described influenza when asked about malaria, and an inaccurate description of how malaria spreads. That specific failure pattern -- real fluency, unreliable facts -- is exactly what fine-tuning on correct, validated data is meant to fix.

**Why the SMALL validated dataset (169 examples), not the larger 1,647-example expanded set:** this model's problem is factual accuracy, not basic fluency. Training it on partly-unreviewed, sometimes-mistranslated data (the expanded set has a known real error: "breastfeeding" mistranslated as "pregnancy") risks teaching it wrong facts more confidently. The small set is 100% human-reviewed -- every example it sees is actually correct.

**Fixes already baked in from real failures on other models** (so we don't repeat them):
- `adamw_torch` instead of the canonical `paged_adamw_8bit` -- the latter caused a real CUDA crash on Colab/Kaggle T4s.
- Conservative batch size (1) / sequence length (512) -- an 8B model has less memory headroom than the smaller models tried before; verified real word counts in this dataset show 512 tokens is enough to not truncate real answers.
- Target modules verified to match this model's REAL layer names (done in the zero-shot notebook) before assuming the standard Llama names apply.
- The adapter is zipped and saved in the SAME cell that finishes training, immediately -- a real earlier run lost its output because a later cell was never reached before the session ended.

## Before you start
1. Visit https://huggingface.co/Jacaranda/YorubaLlama while logged in and click **"Agree and access repository"** if you haven't already (your Kaggle secret should already have access if the zero-shot check notebook worked).
2. In this notebook: **Add-ons -> Secrets -> Add secret**, label `HF_TOKEN`, value = your token, attached to this notebook (reuse the same one from before).
3. Upload `data/final_demo/train.jsonl`, `val.jsonl`, `test.jsonl` (the SMALL validated set, NOT `final_demo_expanded`) as a Kaggle Dataset, and attach it via **Add Input**.
4. Turn on a **GPU** accelerator and switch **Internet** to **On**.
5. **Use "Save Version -> Save & Run All (Commit)", not an interactive Draft session.** A real earlier run's trained files were permanently lost when a Draft session timed out unattended -- a committed run keeps its output no matter what.

In [ ]:
!nvidia-smi

In [ ]:
# Idempotent clone -- a real run showed re-running this inside an already-
# running session creates a nested duplicate folder if one already exists.
import shutil

%cd /kaggle/working
shutil.rmtree("Yoruba-HealthQA", ignore_errors=True)
!git clone https://github.com/ayoolaeni/Yoruba-HealthQA.git
%cd Yoruba-HealthQA

In [ ]:
# Not pinning versions on purpose -- Kaggle's base image drifts over time
# and a pin chosen today can be stale later. Letting pip resolve current
# mutually-compatible versions has been more robust in practice here.
!pip install -q -U transformers peft bitsandbytes accelerate datasets sacrebleu sentencepiece pyyaml

In [ ]:
from huggingface_hub import login

try:
    from kaggle_secrets import UserSecretsClient
    _token = UserSecretsClient().get_secret("HF_TOKEN")
except Exception:
    import getpass
    _token = getpass.getpass("Paste your Hugging Face access token: ")
login(token=_token)

In [ ]:
# SAFETY CHECK: this was already verified once in the zero-shot notebook
# (standard Llama-3 names, all present) -- re-verifying here too rather
# than assuming, since this is a separate session and separate model load.
import torch
from transformers import AutoModelForCausalLM

EXPECTED_TARGET_MODULES = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"]

_probe_model = AutoModelForCausalLM.from_pretrained("Jacaranda/YorubaLlama", torch_dtype=torch.float16)
_linear_suffixes = sorted({name.rsplit(".", 1)[-1] for name, mod in _probe_model.named_modules()
                            if mod.__class__.__name__ == "Linear" and "." in name})
del _probe_model

print("Real Linear-layer name suffixes found:", _linear_suffixes)
missing = [m for m in EXPECTED_TARGET_MODULES if m not in _linear_suffixes]
assert not missing, (
    f"STOP: target_modules={EXPECTED_TARGET_MODULES} assumed, but this model is missing {missing} "
    f"among its real layer names ({_linear_suffixes}). Update TARGET_MODULES in the next cell first."
)
print("OK: matches the standard Llama target_modules assumed below.")

In [ ]:
# Find the attached dataset -- searches at any depth under /kaggle/input/
# since Kaggle's actual nesting depth for an attached dataset has varied
# in practice (seen one level deeper than expected before).
import os
import shutil
from pathlib import Path

REQUIRED_FILES = ["train.jsonl", "val.jsonl", "test.jsonl"]

kaggle_input = Path("/kaggle/input")
found_dir = None
for candidate_dir in [kaggle_input] + [p for p in kaggle_input.rglob("*") if p.is_dir()]:
    if all((candidate_dir / fname).exists() for fname in REQUIRED_FILES):
        found_dir = candidate_dir
        break

assert found_dir is not None, (
    f"Could not find all of {REQUIRED_FILES} anywhere under /kaggle/input/. "
    f"Make sure you attached the SMALL validated dataset (data/final_demo/, NOT final_demo_expanded) "
    f"via 'Add Input' first."
)

os.makedirs("data/final_demo", exist_ok=True)
for fname in REQUIRED_FILES:
    shutil.copy(found_dir / fname, f"data/final_demo/{fname}")
print(f"Copied dataset files from {found_dir} -> data/final_demo/:", os.listdir("data/final_demo"))

In [ ]:
# Configure the run. target_modules is left at the canonical config's
# default (already Llama-style q/k/v/o/gate/up/down_proj) since we just
# verified this model matches it -- no override needed, unlike BLOOM or
# InkubaLM which needed different names entirely.
#
# OPTIMISER: adamw_torch, not the canonical paged_adamw_8bit -- a real run
# on a different model crashed with "CUDA error: an illegal memory access"
# inside bitsandbytes' paged optimiser on a T4. Demo-only override; the
# real dissertation config keeps paged_adamw_8bit as specified.
#
# BATCH SIZE / SEQ LENGTH: kept conservative (1 x 16 accum, 512 tokens) --
# an 8B model has less memory headroom than the smaller models tried
# before, and 512 tokens was already verified sufficient for this
# dataset's real answer lengths (max ~173 words) without truncating.
#
# EPOCH SWEEP [1, 2, 3]: with only 169 training examples (steps/epoch ~11),
# this totals just 66 steps across all 3 candidates -- fast enough to
# actually run the real validation-chrF++ selection this time (unlike the
# larger dataset's 143-example validation set, which made scoring
# impractically slow and had to be skipped).
BASE_MODEL = "Jacaranda/YorubaLlama"
EPOCH_CANDIDATES = [1, 2, 3]
OPTIMISER = "adamw_torch"
PER_DEVICE_TRAIN_BATCH_SIZE = 1
GRADIENT_ACCUMULATION_STEPS = 16
MAX_SEQ_LENGTH = 512

import yaml

with open("configs/train.yaml", encoding="utf-8") as f:
    train_cfg = yaml.safe_load(f)
train_cfg["base_model"] = BASE_MODEL
train_cfg["lora"]["r"]["sweep"] = [train_cfg["lora"]["r"]["initial"]]
train_cfg["optim"]["learning_rate"]["sweep"] = [train_cfg["optim"]["learning_rate"]["initial"]]
train_cfg["optim"]["epochs"]["sweep"] = EPOCH_CANDIDATES
train_cfg["optim"]["optimiser"] = OPTIMISER
train_cfg["optim"]["per_device_train_batch_size"] = PER_DEVICE_TRAIN_BATCH_SIZE
train_cfg["optim"]["gradient_accumulation_steps"] = GRADIENT_ACCUMULATION_STEPS
train_cfg["optim"]["max_seq_length"] = MAX_SEQ_LENGTH
with open("configs/train.yaml", "w", encoding="utf-8") as f:
    yaml.safe_dump(train_cfg, f, sort_keys=False, allow_unicode=True)
print(f"configs/train.yaml base_model={BASE_MODEL}, epoch sweep={EPOCH_CANDIDATES}, optimiser={OPTIMISER}, "
      f"batch_size={PER_DEVICE_TRAIN_BATCH_SIZE} x accum={GRADIENT_ACCUMULATION_STEPS}, max_seq_length={MAX_SEQ_LENGTH}")

In [ ]:
# Train all 3 epoch candidates and select the winner by validation chrF++.
# With only 169 train / 26 val examples this should be minutes, not hours --
# a real, much smaller job than the earlier 1,141-example / 143-val-example
# run that made full validation scoring impractically slow.
# PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True is PyTorch's own
# documented mitigation for the OOM message seen in an earlier real run.
!PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True python scripts/09_train.py --sweep --data-dir data/final_demo \
    --train-config configs/train.yaml --model-config configs/model.yaml \
    --models-dir models --reports-dir reports

In [ ]:
# Show the sweep results, copy the winner, and package it for download --
# ALL in this one cell, immediately after training finishes, so the
# trained files are safely zipped even if something goes wrong afterward
# (a real earlier run lost its output because a later, separate cell was
# never reached before the session ended).
import os
import shutil

import pandas as pd

df = pd.read_csv("reports/table_4_4_hyperparameter_sweep.csv").sort_values("val_chrf++", ascending=False)
print(df[["run", "epochs", "train_loss", "val_chrf++"]].to_string(index=False))

best = df.iloc[0]
winning_run_dir = best["run_dir"]
print(f"\nWinner: epochs={best['epochs']} val_chrf++={best['val_chrf++']:.2f} (train_loss={best['train_loss']:.3f})")

target = "models/single_run"
if os.path.exists(target):
    shutil.rmtree(target)
shutil.copytree(winning_run_dir, target)
print(f"Copied {winning_run_dir} -> {target}")

shutil.make_archive("/kaggle/working/single_run_adapter", "zip", target)
print("DONE. Saved to /kaggle/working/single_run_adapter.zip -- download it from the Output tab "
      "once this committed version finishes.")

## On your own laptop, after downloading

1. Unzip `single_run_adapter.zip` into your project's `models/` folder. Point `YHQA_ADAPTER_PATH` at whichever folder actually contains `adapter_config.json` (check for an extra nested folder from the zip).
2. Set these environment variables:
```
YHQA_BASE_MODEL=Jacaranda/YorubaLlama
YHQA_ADAPTER_PATH=models/single_run
```
3. Because the base model is gated, your laptop needs to be logged in to Hugging Face too: `hf auth login` (or the `huggingface-cli login` equivalent), same token as here.
4. **Licence note:** CC BY-NC-SA 4.0 -- non-commercial, share-alike. Fine for your dissertation; contact Jacaranda Health (ai@jacarandahealth.org) before any commercial use.
5. **Size/speed note:** this is an 8B model -- roughly 16GB to download in full precision (or a few GB in 4-bit, which is what the app loads by default). Expect live generation on CPU to be slow, likely similar to or slower than the ~5 minute/answer seen with the earlier 3.35B model tested -- `YHQA_ANSWER_MODE=retrieval` remains the fast, reliable fallback for the live demo regardless of how this turns out.
6. Also download `reports/table_4_4_hyperparameter_sweep.md` from the file browser -- a real sweep results table for your dissertation.
7. **Before trusting any answer**: actually read the generated output on your own example questions first. The zero-shot check already showed real but inconsistent accuracy (a genuine influenza/malaria mix-up) -- fine-tuning should help, but verify rather than assume it fully fixed that.